In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Evaluating AG2 Multi-Agent Pipeline

This notebook demonstrates how to evaluate the AG2 multi-agent pipeline
(architect -> coder -> reviewer) using Vertex AI Gen AI Evaluation.

## Overview

The AG2 template implements a three-agent collaboration pipeline:
1. **Architect** - Designs the system architecture
2. **Coder** - Implements the design
3. **Reviewer** - Reviews and approves or requests changes

We evaluate the pipeline on response quality and coherence.

In [ ]:
%pip install --upgrade --quiet "ag2[gemini]" "google-cloud-aiplatform[evaluation]"

In [ ]:
import os

import vertexai

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "[your-project-id]")
LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "us-central1")

vertexai.init(project=PROJECT_ID, location=LOCATION)

## Run the AG2 Pipeline

In [ ]:
from app.agent import pattern
from autogen.agentchat import initiate_group_chat

result, final_context, last_agent = initiate_group_chat(
    pattern=pattern,
    messages="Design and implement a REST API for a wiki service with article CRUD operations and search",
    max_rounds=20,
)

print(f"Last agent: {last_agent.name}")
print(f"Messages: {len(result.chat_history)}")
for msg in result.chat_history:
    role = msg.get('name', msg.get('role', 'unknown'))
    content = msg.get('content', '')[:200]
    print(f"\n--- {role} ---")
    print(content)

## Evaluate Response Quality

Use Vertex AI Gen AI Evaluation to assess the pipeline output.

In [ ]:
import pandas as pd
from vertexai.preview.evaluation import EvalTask

# Collect the final response from the pipeline
final_response = result.chat_history[-1].get("content", "") if result.chat_history else ""

eval_data = pd.DataFrame({
    "prompt": [
        "Design and implement a REST API for a wiki service with article CRUD operations and search",
    ],
    "response": [final_response],
})

eval_task = EvalTask(
    dataset=eval_data,
    metrics=["safety", "coherence"],
)

eval_result = eval_task.evaluate()
print(eval_result.summary_metrics)
eval_result.metrics_table